# Convert reconstructed SPECT counts to Bq/mL

This workflow finds reconstructed SPECT DICOM files below a user-selected directory, converts each image from scanner counts to quantitative Bq/mL with `DicomModify.make_bqml_suv`, and writes a new file beside the source using the suffix `_qSPECT.dcm`.

One run represents **one radiopharmaceutical administration**. All time points therefore share the calibration factor, injection record, patient weight and height. Run the notebook separately for data from different administrations. Source files are never modified.

## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from IPython.display import display

from pytheranostics.dicomtools.dicomtools import DicomModify
from pytheranostics.shared.radioactive_decay import get_activity_at_injection

## 2. Configure the input and calibration

Set `input_directory` to the directory containing the reconstructed SPECT time points. Subdirectories are searched recursively. The calibration factor must be validated for the site, camera, collimator, energy windows, isotope, and reconstruction protocol used for these images. Its required unit is **MBq/(counts/s)**.

> **GE SPECT/CT warning:** GE reconstruction can enable **projections multiplication**. When it is enabled together with resolution recovery, reconstructed counts may be scaled by a factor of 4. The calibration factor entered here must have been established using the same reconstruction setting and must therefore account for this factor. Do not apply a separate manual factor of 4 in this workflow. If reconstructed counts are reduced to remain within the 16-bit integer range, `DicomModify.make_bqml_suv` automatically reads GE private DICOM tag `(0011,103B) Pixel Scale` and multiplies the stored pixels by that value before quantification.

In [ ]:
input_directory = Path("/path/to/reconstructed_spect_timepoints")
calibration_factor = 0.0  # MBq/(counts/s); replace with your validated value

# Discovery and output behavior
search_recursively = True
allowed_modalities = {"NM"}  # Add "PT" only if your SPECT data uses that modality.
overwrite_existing = False

## 3. Enter the administration and patient information

Transcribe these values from the administration record. Activities are in MBq, weight is in kg, height is in cm, dates use `YYYYMMDD`, and times use 24-hour `HHMM`. The pre-injection measurement must be at or before injection, and the post-injection measurement must be at or after injection.

In [ ]:
weight_kg = 0.0
height_cm = 0.0

injection_date = "YYYYMMDD"
pre_inj_activity_mbq = 0.0
pre_inj_time = "HHMM"
post_inj_activity_mbq = 0.0
post_inj_time = "HHMM"
injection_time = "HHMM"

radiopharmaceutical = "Lutetium-PSMA-617"
half_life_seconds = 574300.0  # Lu-177
activity_meter_scale_factor = 1.0

## 4. Configure scanner-specific options

For Siemens data, `n_detectors` is multiplied by `NumberOfFramesInRotation` to recover the total projection count. Leave the frame-duration fallback as `None` unless the DICOM `ActualFrameDuration` is missing or invalid and a validated duration is known. The fallback is only applied to Siemens data.

In [ ]:
n_detectors = 2
siemens_frame_duration_fallback = None  # seconds, for example 15.0

## 5. Discover reconstructed SPECT DICOM files

The discovery step reads headers only, selects the configured modalities, and excludes files already ending in `_qSPECT.dcm`. Review the resulting table carefully before conversion. In particular, confirm that every candidate is a reconstructed counts image produced with the calibration-matched protocol.

In [ ]:
def discover_spect_reconstructions(root, recursive=True):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise NotADirectoryError(f"Input directory does not exist: {root}")

    candidates = root.rglob("*") if recursive else root.glob("*")
    records = []
    for path in sorted(candidate for candidate in candidates if candidate.is_file()):
        if path.name.casefold().endswith("_qspect.dcm"):
            continue
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True)
        except Exception:
            continue

        modality = str(getattr(ds, "Modality", "")).upper()
        if modality not in allowed_modalities:
            continue

        pixel_scale_element = ds.get((0x0011, 0x103B))
        records.append(
            {
                "path": path,
                "patient_id": str(getattr(ds, "PatientID", "")),
                "modality": modality,
                "manufacturer": str(getattr(ds, "Manufacturer", "")),
                "series_date": str(getattr(ds, "SeriesDate", "")),
                "series_time": str(getattr(ds, "SeriesTime", "")),
                "series_description": str(getattr(ds, "SeriesDescription", "")),
                "units": str(getattr(ds, "Units", "")),
                "ge_pixel_scale": (
                    str(pixel_scale_element.value) if pixel_scale_element else ""
                ),
            }
        )

    return pd.DataFrame.from_records(records)


spect_reconstructions = discover_spect_reconstructions(
    input_directory, recursive=search_recursively
)
if spect_reconstructions.empty:
    raise FileNotFoundError(
        f"No SPECT DICOM files with modalities {sorted(allowed_modalities)} "
        f"were found below {input_directory}."
    )

display(spect_reconstructions)

## 6. Validate the shared configuration

These checks catch unset placeholders before any output is written. They do not replace verification of the administration record or calibration protocol.

In [ ]:
if calibration_factor <= 0:
    raise ValueError("Set calibration_factor to a positive validated value.")
if weight_kg <= 0:
    raise ValueError("Set weight_kg to a positive value.")
if height_cm <= 0:
    raise ValueError("Set height_cm to a positive value.")
if len(injection_date) != 8 or not injection_date.isdigit():
    raise ValueError("injection_date must use YYYYMMDD.")
for name, value in {
    "pre_inj_time": pre_inj_time,
    "post_inj_time": post_inj_time,
    "injection_time": injection_time,
}.items():
    if len(value) != 4 or not value.isdigit():
        raise ValueError(f"{name} must use HHMM.")
if pre_inj_activity_mbq <= 0 or post_inj_activity_mbq < 0:
    raise ValueError("Pre-injection activity must be positive and post-injection activity cannot be negative.")
if half_life_seconds <= 0:
    raise ValueError("half_life_seconds must be positive.")
patient_ids = {value for value in spect_reconstructions["patient_id"] if value}
if len(patient_ids) != 1:
    raise ValueError(
        "All discovered time points must belong to exactly one patient; "
        f"found patient IDs: {sorted(patient_ids) or 'none'}."
    )

print(f"Configuration validated for {len(spect_reconstructions)} time point(s).")

## 7. Convert all discovered time points

Set `run_conversion = True` only after reviewing the candidate table and configuration. Existing outputs are skipped unless `overwrite_existing` is enabled. An error in one file is recorded in the summary without preventing the remaining time points from being attempted.

In [ ]:
run_conversion = False

if not run_conversion:
    raise RuntimeError(
        "Review the candidate table and configuration, then set "
        "run_conversion = True to create qSPECT files."
    )

results = []
for source_path in spect_reconstructions["path"]:
    output_path = source_path.with_name(f"{source_path.stem}_qSPECT.dcm")
    if output_path.exists() and not overwrite_existing:
        results.append(
            {
                "source_path": source_path,
                "output_path": output_path,
                "status": "skipped",
                "message": "Output already exists",
            }
        )
        continue

    try:
        image = DicomModify(str(source_path), CF=calibration_factor)
        injection_summary = image.make_bqml_suv(
            weight=weight_kg,
            height=height_cm,
            injection_date=injection_date,
            pre_inj_activity=pre_inj_activity_mbq,
            pre_inj_time=pre_inj_time,
            post_inj_activity=post_inj_activity_mbq,
            post_inj_time=post_inj_time,
            injection_time=injection_time,
            activity_meter_scale_factor=activity_meter_scale_factor,
            half_life=half_life_seconds,
            radiopharmaceutical=radiopharmaceutical,
            n_detectors=n_detectors,
            siemens_frame_duration_fallback=siemens_frame_duration_fallback,
        )
        image.ds.save_as(output_path)

        result = injection_summary.iloc[0].to_dict()
        result.update(
            {
                "source_path": source_path,
                "output_path": output_path,
                "status": "converted",
                "message": "",
            }
        )
        results.append(result)
    except Exception as exc:
        results.append(
            {
                "source_path": source_path,
                "output_path": output_path,
                "status": "failed",
                "message": f"{type(exc).__name__}: {exc}",
            }
        )

conversion_summary = pd.DataFrame.from_records(results)
display(conversion_summary)

## 8. Review total activity in the field of view

As a quantitative sanity check, integrate the calibrated Bq/mL values over the entire image volume and compare the result with the activity administered at injection. The total activity in the SPECT field of view would not normally be expected to exceed the injected activity. A value above 100% is flagged for review.

This check is not a substitute for quantitative QC. Unexpected values can indicate an incorrect calibration factor, GE reconstruction scaling mismatch, Pixel Scale problem, acquisition-duration or projection-count error, voxel-geometry error, decay-correction mismatch, contamination, or an incorrect administration record. Conversely, activity below the injected value is expected because of physical decay, biological clearance, and activity outside the field of view.

In [ ]:
def calculate_fov_activity_mbq(dicom_path):
    ds = pydicom.dcmread(dicom_path)
    mapping_sequence = getattr(ds, "RealWorldValueMappingSequence", None)
    if not mapping_sequence:
        raise ValueError(
            f"RealWorldValueMappingSequence is missing from {dicom_path}."
        )

    mapping = mapping_sequence[0]
    slope = float(mapping.RealWorldValueSlope)
    intercept = float(mapping.RealWorldValueIntercept)
    activity_concentration_bqml = (
        ds.pixel_array.astype(np.float64) * slope + intercept
    )

    pixel_spacing_mm = np.asarray(ds.PixelSpacing, dtype=float)
    slice_thickness_mm = float(ds.SliceThickness)
    voxel_volume_ml = float(
        np.prod([pixel_spacing_mm[0], pixel_spacing_mm[1], slice_thickness_mm])
        / 1000.0
    )
    return float(activity_concentration_bqml.sum() * voxel_volume_ml / 1e6)


_, injected_activity_mbq = get_activity_at_injection(
    injection_date=injection_date,
    pre_inj_activity=pre_inj_activity_mbq,
    pre_inj_time=pre_inj_time,
    post_inj_activity=post_inj_activity_mbq,
    post_inj_time=post_inj_time,
    injection_time=injection_time,
    half_life=half_life_seconds,
)
injected_activity_mbq *= activity_meter_scale_factor

fov_activity_records = []
successful_outputs = conversion_summary[
    conversion_summary["status"].isin(["converted", "skipped"])
]
for row in successful_outputs.itertuples(index=False):
    fov_activity_mbq = calculate_fov_activity_mbq(row.output_path)
    percent_injected = 100.0 * fov_activity_mbq / injected_activity_mbq
    fov_activity_records.append(
        {
            "source_path": row.source_path,
            "output_path": row.output_path,
            "fov_activity_MBq": fov_activity_mbq,
            "injected_activity_MBq": injected_activity_mbq,
            "percent_of_injected_activity": percent_injected,
            "exceeds_injected_activity": fov_activity_mbq > injected_activity_mbq,
        }
    )

fov_activity_summary = pd.DataFrame.from_records(fov_activity_records)
display(fov_activity_summary)

if fov_activity_summary.empty:
    print("No converted qSPECT outputs were available for the FOV activity check.")
elif fov_activity_summary["exceeds_injected_activity"].any():
    print(
        "WARNING: At least one qSPECT image contains more total FOV activity "
        "than the calculated injected activity. Review the highlighted values "
        "and quantitative conversion inputs before use."
    )
else:
    print("PASS: No qSPECT image exceeds the calculated injected activity.")

## 9. Confirm completion and perform quality control

The cell below fails if any candidate could not be converted. Before quantitative use, independently confirm the calibration protocol, acquisition timing, projection count, voxel geometry, decay correction, injected activity, quantitative scaling, and image appearance for every output.

In [ ]:
failed = conversion_summary[conversion_summary["status"] == "failed"]
if not failed.empty:
    raise RuntimeError(
        f"{len(failed)} qSPECT conversion(s) failed. Review conversion_summary."
    )

converted_count = int((conversion_summary["status"] == "converted").sum())
skipped_count = int((conversion_summary["status"] == "skipped").sum())
print(f"Completed: {converted_count} converted, {skipped_count} skipped.")